# Business Data Analysis and Reporting Project

## Retail Sales Performance Analysis

**Dataset note:** This project uses a simulated retail sales dataset created for practical learning. It is not actual company data. It includes order dates, regions, sales channels, product categories, units, revenue, cost, and profit. The sample contains intentional missing values and one duplicate order to practise data quality checks.

## Business objective
Evaluate sales performance, identify revenue and profit drivers, compare regions and channels, and provide practical recommendations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("retail_sales_dataset.csv")
print("Dataset shape:", df.shape)
display(df.head())


## 1. Understand business questions and data structure

In [ ]:
print(df.info())
display(df.describe(include="all").T)
print("Missing values:")
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate order IDs:", df["order_id"].duplicated().sum())


## 2. Clean and prepare the data

In [ ]:
clean = df.drop_duplicates(subset=["order_id"]).copy()
clean["order_date"] = pd.to_datetime(clean["order_date"])
for col in ["region", "sales_channel"]:
    clean[col] = clean[col].fillna(clean[col].mode()[0])
clean["month"] = clean["order_date"].dt.to_period("M").astype(str)
clean["profit_margin"] = clean["profit"] / clean["revenue"]
print("Rows before cleaning:", len(df))
print("Rows after removing duplicate orders:", len(clean))
print("Missing values after cleaning:", int(clean.isna().sum().sum()))
display(clean.head())


## 3. Calculate key performance indicators

In [ ]:
total_revenue = clean["revenue"].sum()
total_profit = clean["profit"].sum()
total_orders = clean["order_id"].nunique()
total_units = clean["units_sold"].sum()
average_order_value = total_revenue / total_orders
profit_margin = total_profit / total_revenue

kpis = pd.DataFrame({
    "KPI": ["Total revenue", "Total profit", "Unique orders", "Units sold", "Average order value", "Overall profit margin"],
    "Value": [total_revenue, total_profit, total_orders, total_units, average_order_value, profit_margin]
})
display(kpis)


## 4. Analyze categories, regions, and channels

In [ ]:
print("Revenue by category:")
display(clean.groupby("category")["revenue"].sum().sort_values(ascending=False).to_frame())
print("Profit by category:")
display(clean.groupby("category")["profit"].sum().sort_values(ascending=False).to_frame())
print("Revenue by region:")
display(clean.groupby("region")["revenue"].sum().sort_values(ascending=False).to_frame())
print("Revenue by channel:")
display(clean.groupby("sales_channel")["revenue"].sum().sort_values(ascending=False).to_frame())


## 5. Monthly trend and visualizations

In [ ]:
monthly = clean.groupby(clean["order_date"].dt.to_period("M"))["revenue"].sum()
monthly.index = monthly.index.astype(str)
plt.figure(figsize=(9,4))
monthly.plot(marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month"); plt.ylabel("Revenue")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

for col, title in [
    ("category", "Revenue by Category"),
    ("region", "Revenue by Region"),
    ("sales_channel", "Revenue by Sales Channel")
]:
    values = clean.groupby(col)["revenue"].sum().sort_values(ascending=False)
    plt.figure(figsize=(7,4))
    values.plot(kind="bar")
    plt.title(title); plt.xlabel(col.replace("_"," ").title()); plt.ylabel("Revenue")
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## 6. Business insights and recommendations

In [ ]:
category_revenue = clean.groupby("category")["revenue"].sum().sort_values(ascending=False)
region_revenue = clean.groupby("region")["revenue"].sum().sort_values(ascending=False)
channel_revenue = clean.groupby("sales_channel")["revenue"].sum().sort_values(ascending=False)

print("Top revenue category:", category_revenue.index[0])
print("Top revenue region:", region_revenue.index[0])
print("Top revenue channel:", channel_revenue.index[0])
print("Highest revenue month:", clean.groupby(clean["order_date"].dt.to_period("M"))["revenue"].sum().idxmax())

print("\nRecommendations:")
print("1. Review stock levels for the strongest revenue categories.")
print("2. Investigate lower performing regions and compare their product mix.")
print("3. Compare channel profitability, not only revenue, before shifting marketing spend.")
print("4. Monitor monthly results to plan inventory and promotions.")
print("5. Keep data quality checks in the reporting process.")


## Executive summary

This analysis uses a simulated retail dataset to demonstrate a business reporting workflow. The notebook cleans duplicate and missing records, calculates revenue and profit KPIs, compares product categories, regions, and channels, and visualizes monthly performance. Findings and recommendations should be interpreted as practice results from simulated data, not as claims about a real company.
